In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="There is an imbalance between your GPUs.*",
)

warnings.filterwarnings(
    "ignore",
    message="Was asked to gather along dimension 0.*",
)

warnings.filterwarnings(
    "ignore",
    message="PyTorch is not compiled with NCCL support.*",
)


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
# Finetuning llama 3.2 for sentiment classification

from datasets import load_dataset
import re
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
from transformers import TrainingArguments, Trainer

import numpy
import evaluate
import torch

In [4]:
# GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.set_device(0)
print(f"GPU Device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

GPU Device: NVIDIA GeForce RTX 5090


In [5]:
# Loading the dataset
ds = load_dataset('stanfordnlp/imdb')
_ = ds.pop('unsupervised')
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})

In [6]:
# Printing a few samples
for i in range(10):
    print(f"Sample {i}: {ds['train'][i]}")

Sample 0: {'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few 

In [7]:
# # Filtering out the Links present in each text
# def filter_link(example):
#     example['text'] = re.sub(r'https://\S+','', example['text'])
#     return example

In [8]:
# ds_filtered = ds.map(filter_link)

# # Printing a few samples
# for i in range(10):
#     print(f"Sample {i}: {ds_filtered['train'][i]}")

In [9]:
# Tokenizer
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=1024)

# setting the pad token
tokenizer.pad_token = tokenizer.eos_token


In [10]:
def tokenize(example, tokenizer):
    example = tokenizer(example['text'], padding=False, truncation=True)

    return example

In [11]:
# Checking the number of CPUs
from multiprocessing import cpu_count
cpu_count()

32

In [12]:
tokenized_ds = ds.map(tokenize, batched=True, num_proc=25, 
                               remove_columns=['text',],
                               fn_kwargs={"tokenizer": tokenizer}, )
print(tokenized_ds)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
})


In [13]:
ds_split = tokenized_ds['train'].train_test_split(test_size=0.1,seed=42)
print(ds_split)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 22500
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
})


In [14]:
# Printing a few samples
for i in range(1):
    print(f"Sample {i}: {ds_split['train'][i]}")

Sample 0: {'label': 0, 'input_ids': [128000, 2409, 1521, 1274, 282, 1802, 779, 1690, 15300, 11, 1701, 2362, 22609, 11, 323, 342, 73932, 10099, 311, 636, 1124, 704, 11, 539, 311, 6420, 430, 1063, 315, 279, 16451, 1051, 42508, 389, 264, 3549, 743, 449, 20142, 11, 1148, 596, 311, 4510, 30, 10846, 4632, 315, 5961, 374, 6555, 11, 719, 279, 10065, 11737, 323, 53568, 315, 68473, 374, 26175, 311, 3821, 304, 1521, 12631, 13, 358, 1440, 11, 27052, 374, 10619, 304, 1521, 2362, 12631, 11, 719, 1070, 374, 810, 311, 430, 311, 1304, 420, 5743, 9229, 38769, 13, 5896, 11872, 291, 439, 282, 41476, 11, 814, 2646, 32122, 872, 25761, 11, 11826, 11605, 574, 459, 506, 12, 6723, 78397, 65821, 11, 1511, 4885, 1093, 7762, 7295, 369, 6020, 8895, 1418, 31161, 287, 1124, 315, 85974, 11, 41566, 813, 7555, 596, 10186, 18710, 11, 1701, 1077, 439, 264, 19369, 2047, 11, 682, 420, 3727, 1521, 12631, 99930, 13, 2435, 1051, 555, 912, 3445, 279, 1176, 311, 5944, 311, 1521, 12098, 11, 477, 279, 1176, 311, 3350, 922, 1124, 1

In [15]:
# Loading the datacollator
# It internally calls dataloader of pytorch and converts all the list to pytorch tensors

data_collator = transformers.DataCollatorWithPadding(tokenizer, padding=True)

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, 
                                                           pad_token_id=tokenizer.eos_token_id)
print(model)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((20

In [17]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128001,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.53.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [18]:
model.config.id2label = {0:"Negative",
                         1:"Positive",}

model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "id2label": {
    "0": "Negative",
    "1": "Positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128001,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.53.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [19]:
classifier = TextClassificationPipeline(model=model,
                                       tokenizer=tokenizer,
                                       framework='pt',
                                       task="sentiment-analysis",
                                       device = "cuda"
                                       )

Device set to use cuda


In [20]:
# Model's prediction - before Fine
text = "Very bad movie with no good story" # Has nothing to do with market - should be neutral
prediction = classifier(text)
prediction

[{'label': 'Negative', 'score': 0.9953436255455017}]

In [21]:
# Model's prediction - before Fine
text = "The movie is really bad..nothing new to hook us"# Has nothing to do with market - should be neutral
prediction = classifier(text)
prediction

[{'label': 'Negative', 'score': 0.9528760313987732}]

In [22]:
# Freezing the parameters of all layers expect the last linear layer
for name,param in model.named_parameters():    
    if name != "score.weight":
        param.requires_grad = False
    print(name,param.requires_grad)

model.embed_tokens.weight False
model.layers.0.self_attn.q_proj.weight False
model.layers.0.self_attn.k_proj.weight False
model.layers.0.self_attn.v_proj.weight False
model.layers.0.self_attn.o_proj.weight False
model.layers.0.mlp.gate_proj.weight False
model.layers.0.mlp.up_proj.weight False
model.layers.0.mlp.down_proj.weight False
model.layers.0.input_layernorm.weight False
model.layers.0.post_attention_layernorm.weight False
model.layers.1.self_attn.q_proj.weight False
model.layers.1.self_attn.k_proj.weight False
model.layers.1.self_attn.v_proj.weight False
model.layers.1.self_attn.o_proj.weight False
model.layers.1.mlp.gate_proj.weight False
model.layers.1.mlp.up_proj.weight False
model.layers.1.mlp.down_proj.weight False
model.layers.1.input_layernorm.weight False
model.layers.1.post_attention_layernorm.weight False
model.layers.2.self_attn.q_proj.weight False
model.layers.2.self_attn.k_proj.weight False
model.layers.2.self_attn.v_proj.weight False
model.layers.2.self_attn.o_proj

In [23]:
num_parameters = 0
for param in model.parameters():   
    if param.requires_grad:
        num_parameters += param.numel()
print(f'Number of Parameters:{num_parameters}')

Number of Parameters:4096


In [24]:

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = numpy.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [25]:
training_args = TrainingArguments(output_dir='llma32_imdb_ft',
                                  eval_strategy="steps",
                                  eval_steps=100,
                                  num_train_epochs=1,
                                  per_device_train_batch_size=12,
                                  per_device_eval_batch_size=12,
                                  bf16=False,
                                  fp16=True,
                                  tf32=False,
                                  gradient_accumulation_steps=1,
                                  adam_beta1=0.9,
                                  adam_beta2=0.999,
                                  learning_rate=2e-5,
                                  weight_decay=0.01,
                                  logging_dir='logs',
                                  logging_strategy="steps",
                                  logging_steps = 100,
                                  save_steps=500,
                                  save_total_limit=20,
                                  report_to='none',
                                )

In [26]:
trainer = Trainer(model=model,
                  args = training_args,
                 train_dataset=ds_split["train"],
                 eval_dataset=ds_split["test"],
                 compute_metrics=compute_metrics,
                 data_collator = data_collator)

In [27]:
results = trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,1.866200,0.935598,0.502400
200,0.814700,0.694714,0.638000
300,0.649400,0.571517,0.717600
400,0.538300,0.487465,0.776000
500,0.474300,0.427352,0.810800
600,0.428200,0.395624,0.825200
700,0.401200,0.379146,0.838000
800,0.375200,0.358635,0.846400
900,0.340000,0.343985,0.851600
1000,0.364300,0.335131,0.855200


In [29]:
# Model's prediction - before Fine
text = "Oppenheimer is less a biopic and more a study of moral momentum. Nolan isn’t interested in whether history judges Oppenheimer kindly — he’s interested in how intelligence, once unleashed, stops asking permission. The film moves like a chain reaction: dialogue detonates, timelines collide, and silence becomes as loud as spectacle. It’s a portrait of genius trapped inside consequence."
prediction = classifier(text)
prediction

[{'label': 'Negative', 'score': 0.7884203195571899}]

In [30]:
text= "Blade Runner 2049 is a sequel that understands restraint. Its brilliance lies not in answering the original film’s questions, but in complicating them. Identity here isn’t discovered — it’s manufactured, marketed, and quietly doubted. Villeneuve turns emptiness into atmosphere, proving that loneliness can be cinematic when treated with patience and respect."
prediction = classifier(text)
prediction

[{'label': 'Positive', 'score': 0.6827852725982666}]

In [31]:
text = "Whiplash pretends to be about music, but it’s really about control. The film asks an uncomfortable question: is greatness born from encouragement or cruelty? By refusing to give a clean answer, it forces the viewer to confront their own tolerance for abuse disguised as ambition. The final scene isn’t triumphant — it’s transactional."
prediction = classifier(text)
prediction

[{'label': 'Negative', 'score': 0.8068526387214661}]